Author: HC

version: 0.2

last modifs : 07/09/2025

# Library Import

In [1]:
import pandas as pd
import numpy as np
import os
import glob
import re
from IPython.display import clear_output

# Def parameters

In [2]:
ZT0 = "7:00:00 AM"

dir_input = 'D:\Temp\Sleep_recordings'

annot_dict = {
    "6-16-2025" : "NS",
    "6-27-2025" : "SDW1",
    "7-4-2025" : "SDW2",
    "7-11-2025" : "SDW3"
}

if not os.path.exists(f'{dir_input}/output/'):
    output_dir = os.mkdir(f'{dir_input}/output/')


# Def function

In [3]:
def separate_days(dir_input, ZT0, annot_dict = annot_dict):
    list_files = glob.glob(dir_input + '/' + '*_whole-recording.csv')
    output_dir = f'{dir_input}/output'

    if len(list_files) == 0:
        print('No file matching expected format found, please double check files name ("ID_{mouseIDorwhatever}_whole-recording.csv") and input pathway.')

    for n, file in enumerate(list_files):
        sample = re.split("_", list_files[n])[-2]

        print(f"Start processing of {sample} recording")
        
        df = pd.read_csv(file, low_memory=False)
        df.drop(labels=0, axis=0,inplace=True) ## Remove units
        df = df.reset_index()
        if df.columns[-1] != 'Activity':
            df.drop(labels=df.columns[-1],axis=1,inplace=True)
        if df.columns[0] == 'index':
            df.drop(labels=df.columns[0],axis=1,inplace=True)
        
        unique_dates = df['Time Stamp'].apply(lambda x : x.split(' ')[0])
        unique_dates = unique_dates.unique().astype(str)
        unique_dates_noslash = np.char.replace(unique_dates, "/","-") ## For new filename

        list_pos_CT0 = {}
        for n, date in enumerate(unique_dates):
            print(n, date)
            nex = n+1
            if n==0:
                debut = 0
                temp_position = df[df['Time Stamp']==f'{unique_dates[nex]} {ZT0}'].index[0] - 1
            elif n == len(unique_dates) - 1 :
                debut = df[df['Time Stamp']==f'{unique_dates[n]} {ZT0}'].index[0]
                temp_position = len(df)
            else:
                debut = df[df['Time Stamp']==f'{unique_dates[n]} {ZT0}'].index[0]
                temp_position = df[df['Time Stamp']==f'{unique_dates[nex]} {ZT0}'].index[0] - 1

            list_pos_CT0[unique_dates[n]] = (debut,temp_position)

        list_pos_CT0
        clear_output()

        exp_df = pd.DataFrame(data = {"Date" : unique_dates_noslash,
                              "Expe": np.zeros(len(unique_dates))})
        
        if len(annot_dict) > 0:
            current_exp = ["Undefined"]
            # print(annot_dict)
            for n, date in enumerate(exp_df['Date'].unique()):
                # print(n, date)
                if date in annot_dict.keys():
                    current_exp = annot_dict[date]
                exp_df.loc[n,"Expe"] = current_exp


        pd.options.mode.copy_on_write = True
        for n, key in enumerate(list_pos_CT0.keys()):
            df_temp = df[list_pos_CT0[key][0]:list_pos_CT0[key][1]]
            df_temp.dropna(thresh= 2, axis=0, inplace= True)
            # print(len(df_temp))
            if len(df_temp) > 1:
                if len(annot_dict) > 0:
                    df_temp.to_csv(f'{dir_input}\output\{exp_df.loc[n,"Date"]}_{sample}-{exp_df.loc[n,"Expe"]}.csv',index=None)
                else:
                    df_temp.to_csv(f'{dir_input}\output\{exp_df.loc[n,"Date"]}_{sample}.csv',index=None)
        
    print(" ")
    print('Done')

# Run function

In [5]:
separate_days(dir_input = dir_input,
              ZT0 = ZT0,
            #   annot_dict = annot_dict
              )

 
Done


# Debugging only, please ignore.

In [ ]:
df = pd.read_csv(f'{dir_input}\ID_1267843_whole-recording.csv')

df.drop(labels=0, axis=0,inplace=True) ## Remove units
df = df.reset_index()
if df.columns[-1] != 'Activity':
    df.drop(labels=df.columns[-1],axis=1,inplace=True)
if df.columns[0] == 'index':
    df.drop(labels=df.columns[0],axis=1,inplace=True)

unique_dates = df['Time Stamp'].apply(lambda x : x.split(' ')[0])
unique_dates = unique_dates.unique().astype(str)
unique_dates_noslash = np.char.replace(unique_dates, "/","-") ## For new filename



In [ ]:
df

In [ ]:
list_pos_CT0 = {}
for n, date in enumerate(unique_dates):
    print(n, date)
    nex = n+1
    if n==0:
        debut = 0
        temp_position = df[df['Time Stamp']==f'{unique_dates[nex]} {ZT0}'].index[0] - 1
    elif n == len(unique_dates) - 1 :
        debut = df[df['Time Stamp']==f'{unique_dates[n]} {ZT0}'].index[0]
        temp_position = len(df)
    else:
        debut = df[df['Time Stamp']==f'{unique_dates[n]} {ZT0}'].index[0]
        temp_position = df[df['Time Stamp']==f'{unique_dates[nex]} {ZT0}'].index[0] - 1

    list_pos_CT0[unique_dates[n]] = (debut,temp_position)

list_pos_CT0



In [ ]:
df[df['Time Stamp']==f'{unique_dates[nex]} {ZT0}'].index

In [ ]:
annot_dict = {
    "6-16-2025" : "NS",
    "6-27-2025" : "SDW1",
    "7-4-2025" : "SDW2",
    "7-11-2025" : "SDW3"
}

In [ ]:
current_exp = ["Undefined"]
current_exp = annot_dict['06-16-2025']
current_exp

In [ ]:
annot_df = pd.DataFrame(data= {"Date" : annot_dict.keys(),
                               "Exp" : annot_dict.values()})
print(annot_df)
# unique_dates

In [ ]:
exp_df = pd.DataFrame(data = {"Date" : unique_dates,
                              "Expe": np.zeros(len(unique_dates))})
exp_df

In [ ]:
len(annot_dict)

In [ ]:
current_exp = []
for n, date in enumerate(exp_df['Date'].unique()):
    # print(n, date)
    if date in annot_dict.keys():
        current_exp = annot_dict[date]
    exp_df.loc[n,"Expe"] = current_exp

exp_df

In [ ]:
sample = "test"

In [ ]:
unique_dates_noslash2

In [ ]:
pd.options.mode.copy_on_write = True
for n, key in enumerate(list_pos_CT0.keys()):
    df_temp = df[list_pos_CT0[key][0]:list_pos_CT0[key][1]]
    df_temp.dropna(thresh= 2, axis=0, inplace= True)
    # print(len(df_temp))
    if len(df_temp) > 1:
        df_temp.to_csv(f'{dir_input}\output\{exp_df.loc[n,"Date"]}_{sample}-{exp_df.loc[n,"Expe"]}.csv',index=None)

In [ ]:
unique_dates_noslash